<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 처음부터 구현한 GPT 아키텍처를 Llama 2로 변환하기

- 이 노트북에서는 원본 GPT 아키텍처를 단계별로 Llama 2 모델로 변환합니다 (GPT와 GPT-2는 동일한 아키텍처를 공유함에 주의)
- 왜 Llama 1이나 Llama 3가 아닌가요?
   - Llama 1 아키텍처는 Llama 2와 비슷하지만, Llama 2는 더 큰 컨텍스트 윈도우를 가지고 있습니다 (이는 좋은 점입니다); Llama 1 가중치는 쉽게 사용할 수 없고 더 많은 사용 제한이 있으므로 Llama 2에 집중하는 것이 더 합리적입니다
   - Llama 3에 관해서는, Llama 2를 Llama 3로 변환하는 별도의 노트북을 공유할 예정입니다 (몇 가지 작은 추가 변경사항만 있습니다)
- 이 노트북의 설명은 의도적으로 최소화하여 불필요하게 길어지지 않도록 하고 주요 코드에 집중합니다
- 더 많은 정보는 Llama 2 논문을 참조하세요: [Llama 2: Open Foundation and Fine-Tuned Chat Models (2023)](https://arxiv.org/abs/2307.09288)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

- 이 노트북에서 사용되는 패키지들:

In [ ]:
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # 사전 훈련된 가중치 다운로드용
    "sentencepiece",    # 토크나이저 구현용
    "torch",            # 모델 구현용
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
# 1. GPT 모델 구현을 단계별로 변환하기

- 이 섹션에서는 [4장](../../ch04/01_main-chapter-code/ch04.ipynb)의 GPT 모델 코드를 살펴보고 Llama 2 아키텍처를 구현하기 위해 단계별로 수정합니다
- 나중에 Meta AI에서 공유한 원본 Llama 2 가중치를 로드합니다

&nbsp;
## 1.1 LayerNorm을 RMSNorm 레이어로 교체

- 먼저 LayerNorm을 Root Mean Square Layer Normalization (RMSNorm)으로 교체합니다
- LayerNorm은 평균과 분산을 사용하여 입력을 정규화하는 반면, RMSNorm은 제곱근 평균 제곱(root mean square)만 사용하여 계산 효율성을 개선합니다
- RMSNorm 연산은 다음과 같습니다. 여기서 $x$는 입력이고 $\gamma$는 훈련 가능한 매개변수(벡터)이며, $\epsilon$은 0으로 나누는 오류를 방지하는 작은 상수입니다:

$$y_i = \frac{x_i}{\text{RMS}(x)} \gamma_i, \quad \text{where} \quad \text{RMS}(x) = \sqrt{\epsilon + \frac{1}{n} \sum x_i^2}$$

- 더 자세한 내용은 논문 [Root Mean Square Layer Normalization (2019)](https://arxiv.org/abs/1910.07467)을 참조하세요

In [ ]:
import torch
import torch.nn as nn


#####################################
# 4장
#####################################

# class LayerNorm(nn.Module):
#     def __init__(self, emb_dim):
#         super().__init__()
#         self.eps = 1e-5
#         self.scale = nn.Parameter(torch.ones(emb_dim))
#         self.shift = nn.Parameter(torch.zeros(emb_dim))

#     def forward(self, x):
#         mean = x.mean(dim=-1, keepdim=True)
#         var = x.var(dim=-1, keepdim=True, unbiased=False)
#         norm_x = (x - mean) / torch.sqrt(var + self.eps)
#         return self.scale * norm_x + self.shift


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.emb_dim = emb_dim
        self.weight = nn.Parameter(torch.ones(emb_dim)).float()

    def forward(self, x):
        means = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(means + self.eps)
        return (x_normed * self.weight).to(dtype=x.dtype)

- 다음 코드 셀은 이 구현이 PyTorch의 내장 구현과 동일하게 작동하는지 확인합니다:

In [ ]:
torch.manual_seed(123)

example_batch = torch.randn(2, 3, 4)

rms_norm = RMSNorm(emb_dim=example_batch.shape[-1])
rmsnorm_pytorch = torch.nn.RMSNorm(example_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(example_batch), rmsnorm_pytorch(example_batch))

&nbsp;
## 1.2 GELU를 SiLU 활성화 함수로 교체

- Llama는 (GELU 대신) SiLU 활성화 함수를 사용합니다. 이는 Swish 함수라고도 알려져 있습니다:

$$
\text{silu}(x) = x \cdot \sigma(x), \quad \text{where} \quad \sigma(x) \text{는 로지스틱 시그모이드입니다.}
$$

- 더 많은 정보는 SiLU 논문을 참조하세요: [Sigmoid-Weighted Linear Units for Neural Network Function Approximation in Reinforcement Learning (2017)](https://arxiv.org/abs/1702.03118)

In [ ]:
#####################################
# 4장
#####################################

# class GELU(nn.Module):
#     def __init__(self):
#         super().__init__()

#     def forward(self, x):
#         return 0.5 * x * (1 + torch.tanh(
#             torch.sqrt(torch.tensor(2.0 / torch.pi)) *
#             (x + 0.044715 * torch.pow(x, 3))
#         ))


class SiLU(nn.Module):
    def __init__(self):
        super(SiLU, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
silu = SiLU()

assert torch.allclose(silu(example_batch), torch.nn.functional.silu(example_batch))

&nbsp;
## 1.3 FeedForward 모듈 업데이트

- 실제로 Llama는 SwiGLU라고 불리는 SiLU의 "Gated Linear Unit" (GLU) 변형을 사용하며, 이는 본질적으로 약간 다르게 구조화된 `FeedForward` 모듈을 만듭니다
- SwiGLU는 피드포워드 레이어에서 게이팅 메커니즘을 사용하며, 공식은 다음과 같습니다:

$$\text{SwiGLU}(x) = \text{SiLU}(\text{Linear}_1(x)) * (\text{Linear}_2(x))$$

- 여기서 $\text{Linear}_1$과 $\text{Linear}_2$는 두 개의 선형 레이어이고, $*$는 요소별 곱셈을 나타냅니다
- 세 번째 선형 레이어인 $\text{Linear}_3$는 이 게이트된 활성화 이후에 적용됩니다

- 더 많은 정보는 SwiGLU 논문을 참조하세요: [GLU Variants Improve Transformer (2020)](https://arxiv.org/abs/2002.05202)

In [ ]:
#####################################
# 4장
#####################################
# class FeedForward(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.layers = nn.Sequential(
#             nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
#             GELU(),
#             nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
#         )

#     def forward(self, x):
#         return self.layers(x)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)
        self.silu = SiLU()

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = self.silu(x_fc1) * x_fc2
        return self.fc3(x)

- 위에서 `dtype=cfg["dtype"]` 설정도 추가했음에 주목하세요. 이를 통해 나중에 메모리 사용량을 줄이기 위해 모델을 직접 낮은 정밀도 형식으로 로드할 수 있습니다 (원래 32비트 정밀도 형식으로 인스턴스화한 다음 변환하는 것과 대비)
- Llama는 편향 유닛을 사용하지 않으므로 `bias=False`로 설정했습니다

&nbsp;
## 1.4 RoPE 구현

- GPT 모델에서 위치 임베딩은 다음과 같이 구현됩니다:

```python
self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
```

- 전통적인 절대 위치 임베딩과 달리, Llama는 회전 위치 임베딩(Rotary Position Embeddings, RoPE)을 사용하여 절대 위치 정보와 상대 위치 정보를 동시에 캡처할 수 있습니다
- RoPE의 참조 논문은 [RoFormer: Enhanced Transformer with Rotary Position Embedding (2021)](https://arxiv.org/abs/2104.09864)입니다

In [ ]:
def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # 역주파수 계산
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))

    # 위치 인덱스 생성
    positions = torch.arange(context_length)

    # 각도 계산
    angles = positions[:, None] * inv_freq[None, :]  # 모양: (context_length, head_dim // 2)

    # head_dim에 맞게 각도 확장
    angles = torch.cat([angles, angles], dim=1)  # 모양: (context_length, head_dim)

    # 사인과 코사인 미리 계산
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def compute_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # x를 첫 번째 반과 두 번째 반으로 분할
    x1 = x[..., : head_dim // 2]  # 첫 번째 반
    x2 = x[..., head_dim // 2 :]  # 두 번째 반

    # sin과 cos 모양 조정
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # 모양: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # 회전 변환 적용
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

- 다음은 `q`와 `k` 텐서에 RoPE를 적용하는 예시입니다:

In [ ]:
# 설정
batch_size = 2
context_len = 5
num_heads = 4
head_dim = 16

# RoPE 매개변수 인스턴스화
cos, sin = precompute_rope_params(head_dim=head_dim, context_length=context_len)

# 더미 쿼리와 키 텐서
torch.manual_seed(123)
queries = torch.randn(batch_size, num_heads, context_len, head_dim)
keys = torch.randn(batch_size, num_heads, context_len, head_dim)

# 회전 위치 임베딩 적용
queries_rot = compute_rope(queries, cos, sin)
keys_rot = compute_rope(keys, cos, sin)

&nbsp;
## 1.5 MultiHeadAttention 모듈에 RoPE 추가

- GPT는 입력에 위치 임베딩을 적용하는 반면, Llama는 셀프 어텐션 메커니즘 자체에서 쿼리와 키 벡터에 회전을 적용한다는 점이 중요합니다
- 여기서는 적절한 RoPE 코드로 `MultiHeadAttention` 클래스를 수정합니다
- 또한 `qkv_bias` 옵션을 제거하고 `bias=False` 설정을 하드코딩합니다
- 또한 나중에 더 낮은 정밀도로 모델을 인스턴스화할 수 있도록 dtype 설정을 추가합니다
 - 팁: `TransformerBlock`들(다음 섹션)이 정확히 반복되므로, 각 `MultiHeadAttention` 모듈에 대해 대신 버퍼를 한 번만 초기화하여 코드를 단순화할 수 있습니다. 그러나 독립적인 모듈로 기능할 수 있도록 `MultiHeadAttention` 클래스에 미리 계산된 RoPE 매개변수를 추가합니다

In [ ]:
#####################################
# 3장
#####################################
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dtype=None):  # ,dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 원하는 출력 차원에 맞게 프로젝션 차원 줄이기

        ################################### NEW ###################################
        # 아래 모든 선형 레이어에 대해 bias=False와 dtype=dtype 설정
        ###########################################################################
        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)  # 헤드 출력을 결합하는 선형 레이어
        # self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

        ################################### NEW ###################################
        cos, sin = precompute_rope_params(head_dim=self.head_dim, context_length=context_length)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        ###########################################################################


    def forward(self, x):

        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # 모양: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # `num_heads` 차원을 추가하여 행렬을 암시적으로 분할
        # 마지막 차원 펼치기: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 전치: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        ################################### NEW ###################################
        keys = compute_rope(keys, self.cos, self.sin)
        queries = compute_rope(queries, self.cos, self.sin)
        ###########################################################################

        # 인과 마스크를 사용한 스케일드 닷-프로덕트 어텐션(즉, 셀프 어텐션) 계산
        attn_scores = queries @ keys.transpose(2, 3)  # 각 헤드에 대한 내적

        # 토큰 수에 맞게 잘리고 불린으로 변환된 원본 마스크
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 마스크를 사용하여 어텐션 스코어 채우기
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # attn_weights = self.dropout(attn_weights)

        # 모양: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 헤드 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # 선택적 프로젝션

        return context_vec

- 다음은 예시 입력에서 `MultiHeadAttention` 모듈을 사용하는 예시입니다:

In [ ]:
# 설정
batch_size = 1
context_len = 100
max_context_len = 4096
embed_dim = 128
num_heads = 4


example_batch = torch.randn((batch_size, context_len, embed_dim))

mha = MultiHeadAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=max_context_len,
    num_heads=num_heads
)

mha(example_batch)

del mha  # 메모리 해제를 위한 삭제

&nbsp;
## 1.6 TransformerBlock 모듈 업데이트

- 이 단계에서 대부분의 어려운 작업은 이미 완료되었습니다. 이제 위에서 구현한 코드를 사용하도록 `TransformerBlock`을 업데이트할 수 있습니다
- 이는 다음을 의미합니다:
 - LayerNorm을 RMSNorm으로 교체
 - 드롭아웃 제거
 - `qkv_bias` 설정 제거
 - `dtype` 설정 추가

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dtype=cfg["dtype"]  # NEW
            # dropout=cfg["drop_rate"],
            # qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)

        ################################### NEW ###################################
        # self.norm1 = LayerNorm(cfg["emb_dim"])
        # self.norm2 = LayerNorm(cfg["emb_dim"])
        self.norm1 = RMSNorm(cfg["emb_dim"])
        self.norm2 = RMSNorm(cfg["emb_dim"])
        ###########################################################################

        # self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # 어텐션 블록에 대한 바로가기 연결
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # 모양 [batch_size, num_tokens, emb_size]
        # x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 추가

        # 피드포워드 블록에 대한 바로가기 연결
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        # x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력 다시 추가

        return x

&nbsp;
## 1.7 모델 클래스 업데이트

- [5장](../01_main-chapter-code/ch05.ipynb)에서 기억하실 수 있듯이, `TransformerBlock`은 메인 모델 내에서 반복되는 블록입니다
- 우리의 Llama 모델이 거의 완성되었습니다. `TransformerBlock`을 둘러싼 모델 코드만 업데이트하면 됩니다
- 이는 다음을 의미합니다:
  - 이제 RoPE 임베딩이 있으므로 절대 위치 임베딩 제거
  - LayerNorm을 RMSNorm으로 교체
  - 드롭아웃 제거
  - dtype 설정 추가

In [ ]:
# class GPTModel(nn.Module):
class Llama2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        # self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        # self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        ################################### NEW ###################################
        # self.final_norm = LayerNorm(cfg["emb_dim"])
        self.final_norm = RMSNorm(cfg["emb_dim"])
        ###########################################################################
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

    def forward(self, in_idx):
        # batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        # pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds  # + pos_embeds  # 모양 [batch_size, num_tokens, emb_size]
        # x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

&nbsp;
## 2. 모델 초기화

- 모델 코드가 이제 완성되었고 초기화할 준비가 되었습니다
- [5장](../01_main-chapter-code/ch05.ipynb)에서는 1억 2천 4백만 매개변수 GPT 모델을 지정하기 위해 다음 구성 파일을 사용했습니다:

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # 어휘 크기(vocabulary size)
    "context_length": 1024,  # 컨텍스트 길이(context length)
    "emb_dim": 768,          # 임베딩 차원(embedding dimension)
    "n_heads": 12,           # 어텐션 헤드 수(number of attention heads)
    "n_layers": 12,          # 레이어 수(number of layers)
    "drop_rate": 0.1,        # 드롭아웃 비율(dropout rate)
    "qkv_bias": False        # 쿼리-키-값 편향(query-key-value bias)
}

- 참고로 15억 매개변수 GPT 모델 구성도 아래에 표시됩니다:

In [ ]:
GPT_CONFIG_1558M = {
    "vocab_size": 50257,     # 어휘 크기(vocabulary size)
    "context_length": 1024,  # 컨텍스트 길이(context length)
    "emb_dim": 1600,         # 임베딩 차원(embedding dimension)
    "n_heads": 25,           # 어텐션 헤드 수(number of attention heads)
    "n_layers": 48,          # 레이어 수(number of layers)
    "drop_rate": 0.1,        # 드롭아웃 비율(dropout rate)
    "qkv_bias": False        # 쿼리-키-값 편향(query-key-value bias)
}

- 마찬가지로 7B 모델용 Llama 2 구성 파일을 정의할 수 있습니다 (여기서는 단순성을 위해 다른 더 큰 모델들은 무시합니다):

In [ ]:
LLAMA2_CONFIG_7B = {
    "vocab_size": 32000,     # 어휘 크기(vocabulary size)
    "context_length": 4096,  # 컨텍스트 길이(context length)
    "emb_dim": 4096,         # 임베딩 차원(embedding dimension)
    "n_heads": 32,           # 어텐션 헤드 수(number of attention heads)
    "n_layers": 32,          # 레이어 수(number of layers)
    "hidden_dim": 11008,     # NEW: FeedForward의 중간 차원 크기
    "dtype": torch.bfloat16  # NEW: 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
}

- 이러한 설정을 사용하여 이제 Llama 2 7B 모델을 초기화할 수 있습니다 (이는 약 26GB의 메모리가 필요함에 주의)

In [ ]:
model = Llama2Model(LLAMA2_CONFIG_7B)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

- 위에서 보듯이 모델은 67억 개의 매개변수를 포함합니다 (일반적으로 반올림되어 7B 모델이라고 불림)
- 또한 아래 코드를 사용하여 이 모델의 메모리 요구사항을 계산할 수 있습니다:

In [ ]:
def model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # 매개변수당 총 요소 수 계산
        param_size = param.numel()
        total_params += param_size
        # 이 매개변수에 대해 기울기가 저장되는지 확인
        if param.requires_grad:
            total_grads += param_size

    # 버퍼 크기 계산 (메모리가 필요한 비매개변수)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # 바이트 크기 = (요소 수) * (각 요소의 바이트 크기)
    # 매개변수와 기울기가 입력 dtype과 같은 유형으로 저장된다고 가정
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # 바이트를 기가바이트로 변환
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

- 마지막으로 해당되는 경우 NVIDIA 또는 Apple Silicon GPU로 모델을 전송할 수도 있습니다:

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

&nbsp;
## 3. 토크나이저 로드

- 이 섹션에서는 모델용 토크나이저를 로드할 것입니다
- Llama 2는 OpenAI의 [Tiktoken](https://github.com/openai/tiktoken) 대신 Google의 [SentencePiece](https://github.com/google/sentencepiece) 토크나이저를 사용합니다 (하지만 Llama 3는 Tiktoken을 사용합니다)
- Meta AI는 원본 Llama 2 모델 가중치와 토크나이저 어휘를 Hugging Face Hub에서 공유했습니다
- Hub에서 토크나이저 어휘를 다운로드하여 SentencePiece에 로드할 것입니다
- 필요한 라이브러리를 설치하려면 다음 코드의 주석을 해제하고 실행하세요:

In [ ]:
# !pip install huggingface_hub sentencepiece

- Meta AI는 파일을 다운로드하기 전에 Llama 2 라이선스 조건에 동의해야 합니다. 이를 위해서는 Hugging Face Hub 계정을 만들고 [meta-llama/Llama-2-7b](https://huggingface.co/meta-llama/Llama-2-7b) 저장소를 방문하여 조건에 동의해야 합니다
- 다음으로 액세스 토큰을 만들어야 합니다. 읽기 권한이 있는 액세스 토큰을 생성하려면 우상단의 프로필 사진을 클릭하고 "Settings"를 클릭하세요


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/settings.webp?1" width="300px">

- 그런 다음 액세스 토큰을 만들고 복사하여 다음 코드 셀에 복사하여 붙여넣을 수 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/access-token.webp?1" width="600px">

In [ ]:
from huggingface_hub import login
import json

with open("config.json", "r") as config_file:
    config = json.load(config_file)
    access_token = config["HF_ACCESS_TOKEN"]

login(token=access_token)

- Llama 2 라이선스 조건에 동의했음을 확인하는 데 필요한 액세스 토큰을 통한 로그인 후, 이제 토크나이저 어휘를 다운로드할 수 있습니다:

In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_file = hf_hub_download(
    repo_id="meta-llama/Llama-2-7b",
    filename="tokenizer.model",
    local_dir="Llama-2-7b"
)

- 토크나이저에 더 친숙한 인터페이스를 제공하기 위해 작은 `LlamaTokenizer` 래퍼 클래스를 정의합니다:

In [ ]:
import sentencepiece as spm


class LlamaTokenizer:
    def __init__(self, tokenizer_file):
        sp = spm.SentencePieceProcessor()
        sp.load(tokenizer_file)
        self.tokenizer = sp

    def encode(self, text):
        return self.tokenizer.encode(text, out_type=int)

    def decode(self, ids):
        return self.tokenizer.decode(ids)


tokenizer = LlamaTokenizer(tokenizer_file)

- 이제 `generate` 함수를 사용하여 Llama 2 모델이 새로운 텍스트를 생성하도록 할 수 있습니다:

In [ ]:
from previous_chapters import generate, text_to_token_ids, token_ids_to_text
# `previous_chapters.py` 파일이 로컬에 없는 경우,
# `llms-from-scratch` PyPI 패키지에서 가져올 수 있습니다.
# 자세한 내용은: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 예를 들어,
# from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text



torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- 물론 위에서 볼 수 있듯이 아직 Llama 2 모델을 훈련하지 않았기 때문에 텍스트가 의미가 없습니다
- 다음 섹션에서는 수만에서 수십만 달러의 비용이 드는 직접 훈련 대신 Meta AI의 사전 훈련된 가중치를 로드합니다

&nbsp;
## 4. 사전 훈련된 가중치 로드

- 아래에서는 미세조정 이전의 간단한 텍스트 완성 모델인 ["meta-llama/Llama-2-7b"](https://huggingface.co/meta-llama/Llama-2-7b) 기본 모델을 로드합니다
- 대안으로, 다음 코드 셀의 문자열을 적절히 수정하여 지시 미세조정되고 정렬된 ["meta-llama/Llama-2-7b-chat"](https://huggingface.co/meta-llama/Llama-2-7b-chat) 모델을 로드할 수 있습니다

In [ ]:
weights_file = hf_hub_download(
   repo_id="meta-llama/Llama-2-7b",
   filename="consolidated.00.pth",
   local_dir="Llama-2-7b"
)

In [ ]:
weights = torch.load(weights_file, weights_only=True)

- `weights`에는 다음 텐서들이 포함되어 있습니다 (단순성을 위해 처음 15개만 표시됨):

In [ ]:
list(weights.keys())[:15]

- 다음 함수는 [5장](../01_main-chapter-code/ch05.ipynb)의 `load_weights_into_gpt` 함수를 모델로 하여 사전 훈련된 가중치를 우리의 Llama 2 모델에 로드합니다:

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")

    if isinstance(right, torch.Tensor):
        return torch.nn.Parameter(right.clone().detach())
    else:
        return torch.nn.Parameter(torch.tensor(right))


def permute(w: torch.Tensor, n_heads, out_dim, in_dim):
    return (w.view(n_heads, out_dim // n_heads // 2, 2, in_dim)
             .transpose(1, 2)          # put axis 2 next to heads
             .reshape(out_dim, in_dim))


def load_weights_into_llama(model, param_config, params):

    cfg = LLAMA2_CONFIG_7B
    
    model.tok_emb.weight = assign(model.tok_emb.weight, params["tok_embeddings.weight"])

    for l in range(param_config["n_layers"]):

        # 원본 Meta/Llama 체크포인트는 Q와 K를 저장하여 하나의 복소 RoPE 쌍을 형성하는
        # 두 숫자가 헤드 차원 내에서 서로 옆에 위치합니다 ("슬라이스" 레이아웃).
        # Hugging Face의 것과 유사한 우리의 RoPE 구현은 인터리브된 레이아웃을 기대합니다
        # 예를 들어, n_heads=2이고 head_dim = 8인 경우:
        #                         ┌── pair 0 ──┐      ┌── pair 1 ──┐
        # Meta (sliced):    [ h0:  r0 r1 r2 r3,   h1:  r0 r1 r2 r3  ]
        # Ours & HF (interleaved):  [ h0: r0 r0 r1 r1 r2 r2 r3 r3  , h1: ... ]
        # 더 자세한 정보는 PR에서의 논의를 참조하세요: https://github.com/rasbt/LLMs-from-scratch/pull/747 
        
        # 따라서 아래에서 q_raw와 k_raw에 대해서는 slices_to_interleave 헬퍼를 사용하여
        # 체크포인트 가중치를 다시 정렬해야 합니다

        q_raw = params[f"layers.{l}.attention.wq.weight"]
        model.trf_blocks[l].att.W_query.weight = assign(
            model.trf_blocks[l].att.W_query.weight,
            permute(q_raw, cfg["n_heads"], cfg["emb_dim"], cfg["emb_dim"])
        )
        k_raw = params[f"layers.{l}.attention.wk.weight"]
        model.trf_blocks[l].att.W_key.weight = assign(
            model.trf_blocks[l].att.W_key.weight,
            permute(k_raw, cfg["n_heads"], cfg["emb_dim"], cfg["emb_dim"])
        )
        model.trf_blocks[l].att.W_value.weight = assign(
            model.trf_blocks[l].att.W_value.weight,
            params[f"layers.{l}.attention.wv.weight"]
        )
        model.trf_blocks[l].att.out_proj.weight = assign(
            model.trf_blocks[l].att.out_proj.weight,
            params[f"layers.{l}.attention.wo.weight"]
        )
        model.trf_blocks[l].norm1.weight = assign(
            model.trf_blocks[l].norm1.weight,
            params[f"layers.{l}.attention_norm.weight"]
        )

        # FeedForward 가중치 로드
        model.trf_blocks[l].ff.fc1.weight = assign(
            model.trf_blocks[l].ff.fc1.weight,
            params[f"layers.{l}.feed_forward.w1.weight"]
        )
        # 어떤 이유로 w2와 w3가 가중치 파일에서 잘못된 순서로 제공됨
        model.trf_blocks[l].ff.fc2.weight = assign(
            model.trf_blocks[l].ff.fc2.weight,
            params[f"layers.{l}.feed_forward.w3.weight"]
        )
        model.trf_blocks[l].ff.fc3.weight = assign(
            model.trf_blocks[l].ff.fc3.weight,
            params[f"layers.{l}.feed_forward.w2.weight"]
        )
        model.trf_blocks[l].norm2.weight = assign(
            model.trf_blocks[l].norm2.weight,
            params[f"layers.{l}.ffn_norm.weight"]
        )

    # 출력 레이어 가중치 로드
    model.final_norm.weight = assign(model.final_norm.weight, params["norm.weight"])
    model.out_head.weight = assign(model.out_head.weight, params["output.weight"])


load_weights_into_llama(model, LLAMA2_CONFIG_7B, weights)
model.to(device);

- 다음으로 텍스트 생성을 위해 모델을 사용할 준비가 되었습니다

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort", tokenizer).to(device),
    max_new_tokens=25,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

&nbsp;
## 5. 지시 미세조정 모델 사용

- 앞서 언급했듯이 위에서는 사전 훈련된 기본 모델을 사용했습니다. 지시를 따를 수 있는 모델을 사용하려면 아래와 같이 `"meta-llama/Llama-2-7b-chat"` 모델을 대신 사용하세요

In [ ]:
del model  # 메모리 해제를 위해

weights_file = hf_hub_download(
   repo_id="meta-llama/Llama-2-7b-chat",
   filename="consolidated.00.pth",
   local_dir="Llama-2-7b-chat"
)

weights = torch.load(weights_file, weights_only=True)
model = Llama2Model(LLAMA2_CONFIG_7B)
load_weights_into_llama(model, LLAMA2_CONFIG_7B, weights)
model.to(device);

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("What do llamas eat?", tokenizer).to(device),
    max_new_tokens=25,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

&nbsp;
# 다음 단계는?

- 이 노트북은 원본 GPT-2 아키텍처를 Llama 2 모델로 변환했습니다
- Llama 2를 Llama 3, Llama 3.1, Llama 3.2로 변환하는 방법에 관심이 있으시면 [converting-llama2-to-llama3.ipynb](converting-llama2-to-llama3.ipynb) 노트북을 확인하세요